# Chapter 24: Class Metaprogramming
*From: Fluent Python by Luciano Ramalho*

---

Class metaprogramming is the art of creating or customizing classes at runtime. Python provides a spectrum of tools -- from simple class factories to decorators, `__init_subclass__`, and full metaclasses -- each suited for different levels of complexity.

## TL;DR

- **Classes are first-class objects** with attributes like `__qualname__`, `__subclasses__()`, and `__mro__`.
- **`type(name, bases, dict)`** is a class factory -- it is exactly what Python calls internally for every `class` statement.
- **Class factory functions** call `type()` to build classes dynamically, like `collections.namedtuple`.
- **`__init_subclass__`** (PEP 487) hooks into subclass creation -- validate, register, or enhance subclasses without metaclasses.
- **Class decorators** receive a class, may modify it, and return a class -- `@dataclass` is the prime example.
- **Import time vs runtime:** class bodies, decorators, `__set_name__`, and `__init_subclass__` all run at import time.
- **Metaclasses** (`type` subclasses) can intercept class creation before `type.__new__` -- the only way to configure `__slots__` dynamically.
- **Prefer simpler tools:** `__init_subclass__` > class decorators > metaclasses.

---
## 1. Classes as Objects

Classes in Python are first-class objects. They have attributes, can be passed around, and their class is `type`.

In [ ]:
# Classes are objects, and type is the metaclass
class Dog:
    sound = "woof"

print(f"type(Dog)          = {type(Dog)}")
print(f"type(int)          = {type(int)}")
print(f"Dog.__name__       = {Dog.__name__}")
print(f"Dog.__qualname__   = {Dog.__qualname__}")
print(f"Dog.__bases__      = {Dog.__bases__}")
print(f"Dog.__mro__        = {Dog.__mro__}")

In [ ]:
# __subclasses__() returns current subclasses in memory
class Animal:
    pass

class Cat(Animal):
    pass

class Dog(Animal):
    pass

print(f"Animal subclasses: {Animal.__subclasses__()}")

---
## 2. type() as a Class Factory

`type` has a dual role:
- `type(obj)` returns the class of `obj`
- `type(name, bases, dict)` **creates a new class**

This three-argument form is exactly what Python executes for every `class` statement.

In [ ]:
# Creating a class dynamically with type()
def bark(self):
    return f"{self.name} says woof!"

# This is equivalent to:
# class Dog:
#     sound = "woof"
#     def bark(self): ...
Dog = type("Dog", (object,), {
    "sound": "woof",
    "bark": bark,
    "__init__": lambda self, name: setattr(self, "name", name),
})

rex = Dog("Rex")
print(f"type(rex) = {type(rex)}")
print(f"rex.sound = {rex.sound}")
print(f"rex.bark() = {rex.bark()}")
print(f"Dog.__mro__ = {Dog.__mro__}")

---
## 3. Class Factory Functions

A class factory is a function that creates new classes dynamically by calling `type()`. This pattern is used by `collections.namedtuple`.

In [ ]:
from typing import Any
from collections.abc import Iterator

def record_factory(cls_name, field_names):
    """Create a simple record class with named fields."""
    if isinstance(field_names, str):
        field_names = field_names.replace(",", " ").split()
    field_names = tuple(field_names)

    def __init__(self, *args, **kwargs):
        attrs = dict(zip(self.__slots__, args))
        attrs.update(kwargs)
        for name, value in attrs.items():
            setattr(self, name, value)

    def __iter__(self):
        for name in self.__slots__:
            yield getattr(self, name)

    def __repr__(self):
        values = ", ".join(
            f"{name}={value!r}"
            for name, value in zip(self.__slots__, self)
        )
        return f"{self.__class__.__name__}({values})"

    cls_attrs = {
        "__slots__": field_names,
        "__init__": __init__,
        "__iter__": __iter__,
        "__repr__": __repr__,
    }
    return type(cls_name, (object,), cls_attrs)

# Use the factory
Dog = record_factory("Dog", "name weight owner")
rex = Dog("Rex", 30, "Bob")
print(rex)
print(f"Unpacking: {list(rex)}")

# Records are mutable
rex.weight = 32
print(f"After update: {rex}")

---
## 4. Customizing Subclasses with __init_subclass__

`__init_subclass__` (PEP 487) is called whenever a class is subclassed. It provides a simple hook to validate, register, or enhance subclasses **without metaclasses**.

In [ ]:
# __init_subclass__ for automatic registration
class Plugin:
    """Base class that auto-registers all subclasses."""
    _registry = {}

    def __init_subclass__(cls, /, name=None, **kwargs):
        super().__init_subclass__(**kwargs)
        key = name or cls.__name__.lower()
        Plugin._registry[key] = cls
        print(f"  Registered plugin: {key!r} -> {cls.__name__}")

class JSONPlugin(Plugin, name="json"):
    def load(self, data):
        return f"Loading JSON: {data}"

class CSVPlugin(Plugin, name="csv"):
    def load(self, data):
        return f"Loading CSV: {data}"

class XMLPlugin(Plugin):  # name defaults to "xmlplugin"
    def load(self, data):
        return f"Loading XML: {data}"

print(f"\nRegistry: {Plugin._registry}")

In [ ]:
# __init_subclass__ for field validation
from typing import get_type_hints

class Validated:
    """Base class that validates subclass type annotations."""
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        hints = get_type_hints(cls)
        for name, hint in hints.items():
            if not callable(hint):
                raise TypeError(
                    f"{cls.__name__}.{name}: {hint} is not callable"
                )
        cls._fields = hints

    def __init__(self, **kwargs):
        for name, constructor in self._fields.items():
            value = kwargs.get(name, constructor())
            setattr(self, name, constructor(value))

    def __repr__(self):
        pairs = ", ".join(
            f"{n}={getattr(self, n)!r}" for n in self._fields
        )
        return f"{type(self).__name__}({pairs})"

class Movie(Validated):
    title: str
    year: int
    box_office: float

m = Movie(title="The Godfather", year=1972, box_office=137)
print(m)

# Default values when fields are omitted
m2 = Movie(title="Life of Brian")
print(m2)

---
## 5. Enhancing Classes with Class Decorators

A class decorator is a callable that receives a class, may modify or replace it, and returns a class. Python's `@dataclass` is the most prominent example.

In [ ]:
# Simple class decorator that adds methods
def add_repr(cls):
    """Class decorator that adds a __repr__ method."""
    def __repr__(self):
        attrs = ", ".join(
            f"{k}={v!r}"
            for k, v in self.__dict__.items()
            if not k.startswith("_")
        )
        return f"{cls.__name__}({attrs})"
    cls.__repr__ = __repr__
    return cls

def add_eq(cls):
    """Class decorator that adds __eq__ based on __dict__."""
    def __eq__(self, other):
        if type(self) is not type(other):
            return NotImplemented
        return self.__dict__ == other.__dict__
    cls.__eq__ = __eq__
    return cls

@add_repr
@add_eq
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

p1 = Point(1, 2)
p2 = Point(1, 2)
p3 = Point(3, 4)
print(p1)
print(f"p1 == p2: {p1 == p2}")
print(f"p1 == p3: {p1 == p3}")

---
## 6. Import Time vs Runtime

Python executes top-level code at import time. The evaluation order for class metaprogramming is:

1. **Class body** executes (top to bottom)
2. **`__set_name__`** is called on descriptors
3. **`__init_subclass__`** fires on the parent class
4. **Class decorator** is applied

All of this happens at import time, before any instances are created.

In [ ]:
# Demonstration of evaluation order
class Descriptor:
    def __set_name__(self, owner, name):
        print(f"  2. __set_name__: {owner.__name__}.{name}")
        self.name = name

    def __set__(self, instance, value):
        instance.__dict__[self.name] = value

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.name)

def my_decorator(cls):
    print(f"  4. decorator applied to {cls.__name__}")
    return cls

class Base:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        print(f"  3. __init_subclass__: {cls.__name__}")

print("1. Class body executes...")

@my_decorator
class MyClass(Base):
    attr = Descriptor()  # __set_name__ called after body
    print("  1. Inside class body")

print("5. All done at import time.")

---
## 7. Metaclasses

A metaclass is a subclass of `type` whose instances are classes. The key methods:
- `__prepare__(meta, name, bases)`: returns the mapping for the class namespace
- `__new__(meta, name, bases, namespace)`: creates the class object
- `__init__(cls, name, bases, namespace)`: initializes the class object

**Use sparingly:** metaclasses are the only way to configure `__slots__` dynamically, but `__init_subclass__` and class decorators handle most other use cases.

In [ ]:
# Simple metaclass example
class AutoSlotsMeta(type):
    """Metaclass that auto-creates __slots__ from annotations."""
    def __new__(mcs, name, bases, namespace):
        # Get annotations from the namespace
        annotations = namespace.get("__annotations__", {})
        if annotations and "__slots__" not in namespace:
            namespace["__slots__"] = tuple(annotations)
            # Remove default values that would conflict with slots
            for attr_name in annotations:
                namespace.pop(attr_name, None)
        return super().__new__(mcs, name, bases, namespace)

class SlottedBase(metaclass=AutoSlotsMeta):
    pass

class Coordinate(SlottedBase):
    x: float
    y: float
    z: float

    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z

    def __repr__(self):
        return f"Coordinate({self.x}, {self.y}, {self.z})"

c = Coordinate(1.0, 2.0, 3.0)
print(c)
print(f"__slots__: {Coordinate.__slots__}")
has_dict = hasattr(c, "__dict__")
print(f"Has __dict__? {has_dict}")  # False = memory efficient

In [ ]:
# __prepare__ example: ordered namespace
class OrderedMeta(type):
    """Metaclass that tracks the order of class attribute definitions."""
    @classmethod
    def __prepare__(mcs, name, bases):
        # Return a regular dict (dicts are ordered since Python 3.7)
        return dict()

    def __new__(mcs, name, bases, namespace):
        cls = super().__new__(mcs, name, bases, namespace)
        # Store the defined attributes (excluding dunders)
        cls._field_order = [
            k for k in namespace
            if not k.startswith("_") and not callable(namespace[k])
        ]
        return cls

class Config(metaclass=OrderedMeta):
    host = "localhost"
    port = 8080
    debug = True
    timeout = 30

print(f"Field order: {Config._field_order}")
for field in Config._field_order:
    print(f"  {field} = {getattr(Config, field)}")

---
## 8. When to Use Metaclasses (Rarely)

Modern Python features have replaced most metaclass use cases:

| Need | Use |
|---|---|
| Validate/register subclasses | `__init_subclass__` |
| Name descriptors automatically | `__set_name__` |
| Add methods/modify class | Class decorators |
| Configure `__slots__` dynamically | **Metaclass** (only remaining hard need) |
| Track class namespace ordering | **Metaclass** with `__prepare__` |

**Rules of thumb:**
- A class can have only **one** metaclass
- Metaclasses should be hidden behind a user-friendly base class
- Prefer the simplest tool that works

---
## Try It Yourself

### Exercise 1: Class Factory
Write a `named_class(name, *attrs)` factory that creates a class with `__init__`, `__repr__`, and `__eq__` for the given attributes.

In [ ]:
# Exercise 1
def named_class(cls_name, *attrs):
    def __init__(self, *args):
        for name, val in zip(attrs, args):
            setattr(self, name, val)

    def __repr__(self):
        pairs = ", ".join(
            f"{a}={getattr(self, a)!r}" for a in attrs
        )
        return f"{self.__class__.__name__}({pairs})"

    def __eq__(self, other):
        if type(self) is not type(other):
            return NotImplemented
        return all(
            getattr(self, a) == getattr(other, a) for a in attrs
        )

    return type(cls_name, (object,), {
        "__init__": __init__,
        "__repr__": __repr__,
        "__eq__": __eq__,
    })

Color = named_class("Color", "r", "g", "b")
c1 = Color(255, 128, 0)
c2 = Color(255, 128, 0)
c3 = Color(0, 0, 0)
print(c1)
print(f"c1 == c2: {c1 == c2}")
print(f"c1 == c3: {c1 == c3}")

### Exercise 2: Plugin Registry with __init_subclass__
Create a `Handler` base class where subclasses auto-register by a `handles` keyword argument. Add a `dispatch(event)` classmethod that finds the right handler.

In [ ]:
# Exercise 2
class Handler:
    _dispatch = {}

    def __init_subclass__(cls, /, handles=None, **kwargs):
        super().__init_subclass__(**kwargs)
        if handles:
            for event in (handles if isinstance(handles, (list, tuple)) else [handles]):
                Handler._dispatch[event] = cls

    @classmethod
    def dispatch(cls, event, *args, **kwargs):
        handler_cls = cls._dispatch.get(event)
        if handler_cls is None:
            raise ValueError(f"No handler for {event!r}")
        return handler_cls().handle(*args, **kwargs)

    def handle(self, *args, **kwargs):
        raise NotImplementedError

class ClickHandler(Handler, handles="click"):
    def handle(self, x, y):
        return f"Click at ({x}, {y})"

class KeyHandler(Handler, handles=["keyup", "keydown"]):
    def handle(self, key):
        return f"Key event: {key}"

print(Handler.dispatch("click", 100, 200))
print(Handler.dispatch("keydown", "Enter"))
print(f"Registry: {Handler._dispatch}")

### Exercise 3: Class Decorator that Adds Validation
Write a `@validated` class decorator that wraps `__init__` to type-check arguments based on annotations.

In [ ]:
# Exercise 3
from typing import get_type_hints
import functools

def validated(cls):
    original_init = cls.__init__
    hints = get_type_hints(original_init)
    hints.pop("return", None)  # ignore return annotation

    @functools.wraps(original_init)
    def checked_init(self, *args, **kwargs):
        # Bind args to parameter names
        import inspect
        sig = inspect.signature(original_init)
        bound = sig.bind(self, *args, **kwargs)
        bound.apply_defaults()

        for param_name, value in list(bound.arguments.items())[1:]:  # skip self
            expected = hints.get(param_name)
            if expected and not isinstance(value, expected):
                raise TypeError(
                    f"{param_name} must be {expected.__name__}, "
                    f"got {type(value).__name__}"
                )
        original_init(self, *args, **kwargs)

    cls.__init__ = checked_init
    return cls

@validated
class Person:
    def __init__(self, name: str, age: int):
        self.name = name
        self.age = age

    def __repr__(self):
        return f"Person({self.name!r}, {self.age!r})"

p = Person("Alice", 30)
print(p)

try:
    Person("Bob", "thirty")  # should raise TypeError
except TypeError as e:
    print(f"Caught: {e}")

---
## Key Takeaways

1. **Classes are first-class objects** -- you can create, inspect, modify, and pass them around just like any other value.

2. **`type(name, bases, dict)`** is the fundamental class factory. Every `class` statement calls `type` behind the scenes.

3. **`__init_subclass__`** is the modern, clean hook for customizing subclass creation -- use it for registration, validation, and enhancement.

4. **Class decorators** are simpler than metaclasses for most class modifications. `@dataclass` is the standard library's showcase example.

5. **Import time evaluation order:** class body -> `__set_name__` -> `__init_subclass__` -> class decorator.

6. **Metaclasses** (`type` subclasses) remain necessary only for intercepting class creation before `type.__new__` -- primarily for dynamic `__slots__` configuration.

7. **Prefer simplicity:** `__init_subclass__` > class decorators > metaclasses. Only reach for more power when simpler tools cannot do the job.

---
## See Also

- [[classes-as-objects]] -- Classes as first-class objects and type()
- [[class-factory-functions]] -- Building classes dynamically with type()
- [[init-subclass]] -- Customizing subclass creation with __init_subclass__
- [[class-decorators]] -- Enhancing classes with decorators
- [[import-time-vs-runtime]] -- When metaprogramming code runs
- [[metaclasses]] -- Metaclass __new__, __init__, and __prepare__
- [[when-to-use-metaclasses]] -- Guidelines for choosing the right tool

**This concludes the Fluent Python class metaprogramming chapters.**